In [6]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [7]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [13]:
# Time
start = dt.datetime(2019,5,5)
end = dt.datetime(2019,5,6)
print(start,end,end-start)

2019-05-05 00:00:00 2019-05-06 00:00:00 1 day, 0:00:00


In [18]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}} ,
                              {"created_at","login_details.last_request_at","sign_up_details.app.platform"}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)

df = df[df["sign_up_details_app_platform"] == "UNITY_Android"]
#df = df[df["sign_up_details_device_id"].isin(devices)]
users = df[["_id","created_at","login_details_last_request_at"]]
users.columns = ["user_id","createtime","last_request"]
print(len(users))
users.head()

107


,user_id,createtime,last_request
0,5cce340cf4adb47a3de944ec,2019-05-05 00:53:32.178,2019-05-06 17:04:35.925
1,5cce3f46a6267c79b83592b7,2019-05-05 01:41:26.328,2019-05-05 01:54:19.874
2,5cce41f0f4adb47a3de96f00,2019-05-05 01:52:48.406,2019-05-05 01:53:35.919
3,5cce431da05eef79974e15ab,2019-05-05 01:57:49.905,2019-05-10 14:34:44.946
4,5cce44af2321b225a12fc2d9,2019-05-05 02:04:31.491,2019-05-05 03:01:46.833


In [26]:
c_payment = cursor.superstars.payment_orders
aw = []
#for documents in c_payment.find({'status':2,'created_at':{'$gte': start}}):
for documents in c_payment.find({'status':2},):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
pay = pd.DataFrame(dic_flattened)
pay = pay.rename(columns={'revenue_top_line':'money'})
pay = pay[(pay["user_name"] != "FirzenYogesh")&(pay["user_name"] != "MrYoBear")&(pay["user_name"] != "goyaala")&(pay['money']>0)]
pay.sort_values('money',ascending=False,inplace=True)
pay.head()

,__v,_id,created_at,currency_code,details_gateway,details_misc_bankcode,details_misc_gatewayname,details_misc_payment_method,details_order_id,details_returned_params,...,items_0_id,items_0_type,revenue_bottom_line,money,status,type,updated_at,user_email,user_id,user_name
11,0,5c95aa7068feb84eb34541f2,2019-03-23 03:39:28.025,NaN,google_in_app_billing,NaN,NaN,NaN,GPA.3362-6285-3005-10303,"{""receipt"":""{\""Store\"":\""GooglePlay\"",\""Transa...",...,6,HARD_CURRENCY_PACKAGE,5530.0,7900.0,2,GENERAL_PURCHASE,2019-03-23 03:39:29.311,hw+881@hws.com,5c959dd9762d8817e5282b26,Wall E
87,0,5ccb793df2bcf7711bef1020,2019-05-02 23:11:57.906,INR,google_in_app_billing,NaN,NaN,NaN,GPA.3358-2856-2709-78598,"{""receipt"":""{\""Store\"":\""GooglePlay\"",\""Transa...",...,6,HARD_CURRENCY_PACKAGE,5530.0,7900.0,2,GENERAL_PURCHASE,2019-05-02 23:11:58.841,hw+1556389932376@hws.com,5cc4a02cd3d9160b131fa69a,Flintstone
116,0,5ce386ef62335450027acb24,2019-05-21 05:04:47.527,INR,google_in_app_billing,NaN,NaN,NaN,GPA.3379-3111-9047-46399,"{""receipt"":""{\""Store\"":\""GooglePlay\"",\""Transa...",...,4,HARD_CURRENCY_PACKAGE,1119.3,1599.0,2,GENERAL_PURCHASE,2019-05-21 05:04:48.504,hw+1557665755698@hws.com,5cd817db62ebdd106380c840,parshu
107,0,5cd89eaf62ebdd1063851e0f,2019-05-12 22:31:11.573,INR,google_in_app_billing,NaN,NaN,NaN,GPA.3354-0131-3743-37195,"{""receipt"":""{\""Store\"":\""GooglePlay\"",\""Transa...",...,4,HARD_CURRENCY_PACKAGE,1119.3,1599.0,2,GENERAL_PURCHASE,2019-05-12 22:31:12.157,hw+1557567186030@hws.com,5cd696d2c513ef678e74c36d,Optimus
109,0,5cd96a9c26267106967efc1b,2019-05-13 13:01:16.405,INR,google_in_app_billing,NaN,NaN,NaN,GPA.3399-2178-2236-44033,"{""receipt"":""{\""Store\"":\""GooglePlay\"",\""Transa...",...,4,HARD_CURRENCY_PACKAGE,1119.3,1599.0,2,GENERAL_PURCHASE,2019-05-13 13:01:16.571,hw+1301@hws.com,5ca1a83dc0a01e6c61ee39d1,Jon Snow


In [ ]:
len(pay)

In [28]:
df1 = pd.merge(users,pay,on='user_id',how = 'outer')
#df1.drop(['device_id','flag','createtime','last_request'], axis=1,inplace = True)

df1 = df1[(df1['createtime']-df1['created_at'])< ("30 days, 00:00:00")]
print(len(df1))
total = df1['money'].sum()
print(total)
print(total/len(users))

0
0.0
0.0
